# Overlap Matrix

The overlap matrix is a defining component of a DeepH data point in an atomic-orbital basis:

$$
S_{i\alpha, j\beta}(\boldsymbol{R}_j-\boldsymbol{R}_i) = \int d^3\boldsymbol{r}\, \phi^{*}_{i\alpha}(\boldsymbol{r}-\boldsymbol{R}_i-\mathcal{R}_i) \phi_{j\beta}(\boldsymbol{r}-\boldsymbol{R}_j-\mathcal{R}_j)
$$

`deepx-dock` computes overlap matrices through HPRO for both SIESTA and OpenMX basis files. The output files are `POSCAR`, `info.json`, and `overlap.h5`.


## CLI

The command format is the same for SIESTA and OpenMX. `path_to_poscar` is a VASP POSCAR file, and `path_to_basis_files` is a directory containing the basis files.

### SIESTA

For the case where spin-orbit coupling (SOC) is present:

```bash
dock compute overlap path_to_poscar path_to_basis_files siesta -s
```

For the spinless case:

```bash
dock compute overlap path_to_poscar path_to_basis_files siesta
```

### OpenMX

For the case where spin-orbit coupling (SOC) is present:

```bash
dock compute overlap path_to_poscar path_to_basis_files openmx -s
```

For the spinless case:

```bash
dock compute overlap path_to_poscar path_to_basis_files openmx
```


## Basis Inputs

### SIESTA

`path_to_basis_files` should contain one SIESTA `.ion` file per element in the POSCAR.

### OpenMX

`path_to_basis_files` should contain the OpenMX 3.9 / DFT_DATA19 `.pao` files and a `basis_info.json` file. The JSON file specifies the actual basis selection for each element, for example:

```json
{
  "Mg": "Mg9.0-s3p2d1",
  "P": "P7.0-s2p2d1"
}
```

HPRO reads the OpenMX PAO source radial functions from the OpenMX logarithmic source grid and prepares the radial representation used by HPRO two-center overlap integration. No OpenMX species HDF5 file is used by this overlap command.


## Batch Processing

The original tiered data-directory design is retained. If `--tier-num` is provided, `PATH` is treated as a data directory and each selected data point must contain a `POSCAR`.

```bash
dock compute overlap data_root path_to_basis_files openmx -t 0 -j 4
dock compute overlap data_root path_to_basis_files siesta -t -1
```

`-t -1` processes `data_root` itself, `-t 0` processes `data_root/<data_dirs>`, and larger tier numbers add more directory levels. `-j -1` uses all cores according to the DeepH-dock parallel helper.


## Output Files

By default, output is written next to the input POSCAR. Use `--output-dir` in single-POSCAR mode to write elsewhere.

```text
output_dir/
|-- POSCAR
|-- info.json
`-- overlap.h5
```

`overlap.h5` stores block-sparse atom-pair matrices with datasets `atom_pairs`, `chunk_shapes`, `chunk_boundaries`, and `entries`.


## Python API

Use `save_overlap_from_files` to compute and write DeepH files, or `calc_overlap_from_files` if only the returned HPRO `MatAO` object is needed.


In [ ]:
from deepx_dock.compute.overlap.overlap import calc_overlap_from_files, save_overlap_from_files

# SIESTA
# overlaps = save_overlap_from_files(
#     'path_to_poscar',
#     'path_to_basis_files',
#     'siesta',
#     output_dir='overlap_siesta',
#     spinful=False,
# )

# OpenMX
# overlaps = save_overlap_from_files(
#     'path_to_poscar',
#     'path_to_basis_files',
#     'openmx',
#     output_dir='overlap_openmx',
#     spinful=False,
# )


## Numerical Options

The CLI exposes HPRO reciprocal radial-grid controls when a workflow needs tighter or code-matched integration parameters:

```bash
dock compute overlap POSCAR basis_dir openmx --ecut 1800 --kdense 15
```

Defaults are `--ecut 50` for SIESTA and `--ecut 1800 --kdense 15` for OpenMX. SIESTA leaves `--kdense` unset unless explicitly provided, so HPRO uses its configured default.
